# Phân tích dữ liệu & đặc trưng — Fruit Variety Classification

Notebook kiểu **Kaggle EDA** trước khi train: kiểm tra chất lượng dữ liệu, mất cân bằng lớp, phân bố kích thước/màu, đánh giá tách biệt đặc trưng (PCA/t-SNE), và baseline đánh giá mô hình SVM giống `fruit_project/train.py`.

**Chạy:** mở thư mục gốc project `Digital_Image_Processing` trong Jupyter/VS Code, hoặc đặt `os.chdir` ở cell dưới cho đúng đường dẫn.

## 1. Cấu hình đường dẫn & import

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import display

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Gốc project (thư mục chứa fruit_project/ và data/)
ROOT = Path.cwd()
if not (ROOT / "fruit_project").is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from fruit_project.config import DATA_DIR, META_PATH, MODEL_PATH, ROOT as CFG_ROOT
ROOT = CFG_ROOT
DATA_DIR = ROOT / "data" / "fruits"

from fruit_project.features import extract_features, DIPOptions, preprocess_for_fruit, dip_preprocess

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 11

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR, "exists:", DATA_DIR.is_dir())

ROOT: D:\Digital_Image_Processing
DATA_DIR: D:\Digital_Image_Processing\data\fruits exists: True


## 2. Thu thập metadata từng ảnh

Đọc đệ quy `data/fruits/<lớp>/...` — cùng logic mở rộng file với `train.py`.

In [2]:
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def iter_images(data_dir: Path) -> list[tuple[Path, str]]:
    rows: list[tuple[Path, str]] = []
    if not data_dir.is_dir():
        return rows
    for class_dir in sorted(p for p in data_dir.iterdir() if p.is_dir()):
        label = class_dir.name
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in EXTS:
                rows.append((p, label))
    return rows


def image_stats(path: Path) -> dict | None:
    bgr = cv2.imread(str(path))
    if bgr is None:
        return None
    h, w = bgr.shape[:2]
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return {
        "path": str(path.relative_to(ROOT)),
        "label": None,  # set outside
        "width": w,
        "height": h,
        "aspect": w / (h + 1e-9),
        "mean_gray": float(np.mean(gray)),
        "std_gray": float(np.std(gray)),
        "mean_h": float(np.mean(hsv[:, :, 0])),
        "mean_s": float(np.mean(hsv[:, :, 1])),
        "mean_v": float(np.mean(hsv[:, :, 2])),
    }


pairs = iter_images(DATA_DIR)
records: list[dict] = []
failed: list[str] = []
for p, lab in pairs:
    st = image_stats(p)
    if st is None:
        failed.append(str(p))
        continue
    st["label"] = lab
    records.append(st)

df_meta = pd.DataFrame(records)
print(f"Ảnh hợp lệ: {len(df_meta)}, đọc lỗi: {len(failed)}")
if failed[:5]:
    print("Ví dụ lỗi:", failed[:5])
df_meta.head()

Ảnh hợp lệ: 41302, đọc lỗi: 2298
Ví dụ lỗi: ['D:\\Digital_Image_Processing\\data\\fruits\\Plum\\PlumpÇé1.png', 'D:\\Digital_Image_Processing\\data\\fruits\\Plum\\PlumpÇé10.png', 'D:\\Digital_Image_Processing\\data\\fruits\\Plum\\PlumpÇé100.png', 'D:\\Digital_Image_Processing\\data\\fruits\\Plum\\PlumpÇé1000.png', 'D:\\Digital_Image_Processing\\data\\fruits\\Plum\\PlumpÇé1001.png']


,path,label,width,height,aspect,mean_gray,std_gray,mean_h,mean_s,mean_v
0,data\fruits\Apple\Apple A\Apple 1.png,Apple,480,322,1.490683,122.567598,64.739524,44.286827,89.093672,146.522198
1,data\fruits\Apple\Apple A\Apple 10.png,Apple,480,322,1.490683,116.346435,74.384066,50.639182,94.270426,139.986206
2,data\fruits\Apple\Apple A\Apple 100.png,Apple,480,322,1.490683,135.981962,53.421609,38.161219,79.289460,162.219481
3,data\fruits\Apple\Apple A\Apple 101.png,Apple,480,322,1.490683,123.111665,71.688697,48.901029,93.331179,146.013270
4,data\fruits\Apple\Apple A\Apple 102.png,Apple,480,322,1.490683,91.591304,63.291153,48.652614,100.227187,112.116531


## 3. Số mẫu theo lớp (cân bằng / mất cân bằng)

Nếu chênh lệch lớn → có thể bias model hoặc cần `class_weight` / oversampling (project đã dùng `class_weight='balanced'` trong SVM).

In [ ]:
if df_meta.empty:
    raise RuntimeError("Không có dữ liệu trong data/fruits — thêm ảnh vào các thư mục lớp.")

counts = df_meta["label"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(max(8, len(counts) * 0.4), 5))
counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
ax.set_title("Số ảnh theo từng lớp")
ax.set_xlabel("Lớp")
ax.set_ylabel("Số ảnh")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

ratio = counts / counts.sum()
print("Tỷ lệ % theo lớp:")
display((ratio * 100).round(2))
imbalance_ratio = counts.max() / counts.min()
print(f"\nTỷ lệ max/min giữa các lớp: {imbalance_ratio:.2f} (>3 thường coi là lệch đáng kể)")

## 4. Phân bố kích thước & tỷ lệ khung hình

Ảnh quá nhỏ hoặc tỷ lệ khác biệt nhiều so với **center-crop 128×128** trong `features.py` có thể làm đặc trưng kém ổn định.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df_meta["width"].hist(bins=40, ax=axes[0], color="coral", edgecolor="white")
axes[0].set_title("Chiều rộng (px)")
df_meta["height"].hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_title("Chiều cao (px)")
df_meta["aspect"].hist(bins=40, ax=axes[2], color="mediumpurple", edgecolor="white")
axes[2].set_title("Aspect ratio (W/H)")
plt.tight_layout()
plt.show()

print(df_meta[["width", "height", "aspect"]].describe().round(2))

## 5. Thống kê màu (HSV mean) theo lớp — boxplot

Giúp nhận biết: hai lớp có phân bố **Hue/S/V** chồng lấn nhiều → dễ nhầm (cần texture/shape hoặc thêm dữ liệu đa nền).

In [ ]:
melt_cols = ["mean_h", "mean_s", "mean_v"]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, melt_cols):
    sns.boxplot(data=df_meta, x="label", y=col, ax=ax)
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### 5b. Lưới ảnh mẫu (mỗi lớp 1 ảnh ngẫu nhiên)

Giúp **nhìn trực quan** nền, góc chụp, độ sáng — các yếu tố gây *domain shift* so với ảnh test mới.

In [ ]:
RND = 42
classes = sorted(df_meta["label"].unique())
n_show = min(len(classes), 8)
ncols = max(1, (n_show + 1) // 2)
nrows = 2 if n_show > ncols else 1
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.8 * nrows))
axes = np.atleast_1d(axes).ravel()
slot = 0
for lab in classes[:n_show]:
    sub = df_meta[df_meta["label"] == lab]
    row = sub.sample(1, random_state=RND).iloc[0]
    bgr = cv2.imread(str(ROOT / row["path"]))
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    axes[slot].imshow(rgb)
    axes[slot].set_title(lab, fontsize=9)
    axes[slot].axis("off")
    slot += 1
for j in range(slot, len(axes)):
    axes[j].axis("off")
plt.suptitle("Mẫu ngẫu nhiên mỗi lớp (cố định seed)")
plt.tight_layout()
plt.show()

## 6. Trích vector đặc trưng (giống pipeline train)

Cấu trúc vector (~123 chiều nếu có LBP): **HSV hist (96) + LAB stats (4) + LBP (11) + shape (12)**.

Tham số `MAX_PER_CLASS` giới hạn để notebook chạy nhanh; đặt `None` để dùng hết ảnh.

In [ ]:
import random

MAX_PER_CLASS = 400  # None = tất cả
SEED = 42
rng = random.Random(SEED)

opt = DIPOptions()

by_class: dict[str, list[Path]] = {}
for _, row in df_meta.iterrows():
    p = ROOT / row["path"]
    by_class.setdefault(row["label"], []).append(p)

X_list: list[np.ndarray] = []
y_list: list[str] = []
for lab, paths in sorted(by_class.items()):
    ps = paths.copy()
    rng.shuffle(ps)
    if MAX_PER_CLASS is not None:
        ps = ps[:MAX_PER_CLASS]
    for p in ps:
        bgr = cv2.imread(str(p))
        if bgr is None:
            continue
        feat = extract_features(bgr, dip_options=opt, apply_dip=True)
        X_list.append(feat)
        y_list.append(lab)

X = np.stack(X_list)
y = np.array(y_list)
FEATURE_DIM = X.shape[1]
print("X shape:", X.shape, "feature_dim:", FEATURE_DIM)


def feature_blocks(dim: int) -> dict[str, tuple[int, int]]:
    """Slice theo extract_features: HSV 96 + LAB 4 + (LBP 11) + shape 12."""
    if dim == 123:
        return {
            "HSV hist": (0, 96),
            "LAB stats": (96, 100),
            "LBP": (100, 111),
            "Shape": (111, 123),
        }
    if dim == 112:
        return {
            "HSV hist": (0, 96),
            "LAB stats": (96, 100),
            "LBP": (100, 100),
            "Shape": (100, 112),
        }
    raise ValueError(
        f"Chiều đặc trưng {dim} không khớp 112 (không LBP) hoặc 123 (có LBP)."
    )


BLOCKS = feature_blocks(FEATURE_DIM)
print("Các khối:", BLOCKS)

## 7. Chuẩn hóa theo scaler & PCA 2D

PCA trên **StandardScaler** (giống bước đầu trong `Pipeline` khi train SVM).

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

scaler = StandardScaler()
Xs = scaler.fit_transform(X)
le = LabelEncoder()
y_enc = le.fit_transform(y)

pca = PCA(n_components=2, random_state=SEED)
Z = pca.fit_transform(Xs)

fig, ax = plt.subplots(figsize=(9, 6))
for i, cname in enumerate(le.classes_):
    m = y_enc == i
    ax.scatter(Z[m, 0], Z[m, 1], s=12, alpha=0.6, label=cname)
ax.set_title("PCA 2D (StandardScaler + PCA)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

pca_full = PCA(random_state=SEED)
pca_full.fit(Xs)
cum = np.cumsum(pca_full.explained_variance_ratio_)
n90 = int(np.searchsorted(cum, 0.90) + 1)
print(f"Số thành phần để đạt ~90% phương sai tích lũy: {n90}")

## 8. t-SNE (tùy chọn — chậm hơn PCA)

Chạy trên **subset** hoặc sau PCA (ví dụ 50 thành phần) để giảm thời gian.

In [ ]:
from sklearn.manifold import TSNE

N_TSNE = min(2000, Xs.shape[0])
idx = np.random.RandomState(SEED).choice(Xs.shape[0], N_TSNE, replace=False)
pca50 = PCA(n_components=min(50, Xs.shape[1]), random_state=SEED)
Xs50 = pca50.fit_transform(Xs[idx])

tsne = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=30, random_state=SEED)
T = tsne.fit_transform(Xs50)

fig, ax = plt.subplots(figsize=(9, 6))
ye = y_enc[idx]
for i, cname in enumerate(le.classes_):
    m = ye == i
    ax.scatter(T[m, 0], T[m, 1], s=10, alpha=0.65, label=cname)
ax.set_title(f"t-SNE (n={N_TSNE}, sau PCA 50D)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Độ mạnh từng khối đặc trưng (ANOVA F-score)

Đo mức độ các **nhóm chiều** (màu vs texture vs shape) có liên quan tới nhãn. Đây là gợi ý chẩn đoán, không thay thế độ quan trọng trong SVM RBF.

In [ ]:
from sklearn.feature_selection import f_classif

F, pval = f_classif(Xs, y_enc)
rows = []
for name, (a, b) in BLOCKS.items():
    if a >= b:
        continue
    rows.append(
        {
            "block": name,
            "mean_F": float(np.mean(F[a:b])),
            "sum_F": float(np.sum(F[a:b])),
        }
    )
df_f = pd.DataFrame(rows).sort_values("mean_F", ascending=False)
display(df_f)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=df_f, x="block", y="mean_F", ax=ax, palette="muted")
ax.set_title("Trung bình F-score theo khối đặc trưng")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 10. Baseline SVM + ma trận nhầm lẫn (giống train)

Train/validation split để xem lớp nào dễ nhầm — **cùng kernel RBF + StandardScaler** như `train.py`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

clf = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "svc",
            SVC(
                kernel="rbf",
                C=10.0,
                gamma="scale",
                probability=True,
                class_weight="balanced",
            ),
        ),
    ]
)
clf.fit(X_tr, y_tr)
y_pred = clf.predict(X_va)

print(classification_report(y_va, y_pred))

labels_order = sorted(np.unique(y))
cm = confusion_matrix(y_va, y_pred, labels=labels_order)
fig, ax = plt.subplots(figsize=(max(8, len(labels_order) * 0.6), max(6, len(labels_order) * 0.5)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_order, yticklabels=labels_order, ax=ax)
ax.set_xlabel("Dự đoán")
ax.set_ylabel("Thực tế")
ax.set_title("Confusion matrix (validation 20%)")
plt.tight_layout()
plt.show()

## 11. So sánh với model đã lưu (nếu có)

Nếu đã chạy `python -m fruit_project.train`, file `weights/fruit_svm.joblib` tồn tại — đánh giá trên **cùng tập ảnh** đã nạp ở trên (hoặc báo không tìm thấy).

In [ ]:
import joblib

if MODEL_PATH.is_file():
    bundle = joblib.load(MODEL_PATH)
    model = bundle["model"]
    le_saved = bundle["label_encoder"]
    y_hat = le_saved.inverse_transform(model.predict(X))
    # Độ chính xác trên toàn bộ mẫu notebook (có thể optimistic nếu trùng train)
    acc = float(np.mean(y_hat == y))
    print(f"Model đã lưu — accuracy trên mẫu EDA hiện tại: {acc:.4f}")
    if META_PATH.is_file():
        print("Meta:", json.loads(META_PATH.read_text(encoding="utf-8")))
    cm2 = confusion_matrix(y, y_hat, labels=le_saved.classes_)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm2, annot=True, fmt="d", cmap="Greens", xticklabels=le_saved.classes_, yticklabels=le_saved.classes_, ax=ax)
    ax.set_title("Confusion matrix — model đã train (trên mẫu EDA; có thể overlap train)")
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có model tại", MODEL_PATH, "— bỏ qua bước này hoặc train trước.")

## 12. Tóm tắt vấn đề thường gặp của project

| Hiện tượng | Gợi ý |
|------------|--------|
| Lớp lệch số lượng | Dùng `class_weight`, augment, hoặc cân bằng dữ liệu |
| PCA/t-SNE: lớp chồng lấn | Tăng dữ liệu đa nền/ánh sáng; kiểm tra ROI (YOLO vs center crop) |
| Ảnh test khác miền (webcam, nền mới) | Domain shift — augment khi train, fusion YOLO+SVM, chuẩn hóa ánh sáng (CLAHE) |
| F-score khối HSV >> shape | Model phụ thuộc màu — dễ lỗi khi đèn đổi; cần thêm mẫu hoặc trọng số |

---

**Gợi ý:** sau khi chỉnh augment / ROI mode, chạy lại notebook và so sánh confusion matrix + PCA.